# Movie Recommender System

A cleaned version of the classroom content-based recommender example.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("../data/imdb_top_1000.csv")
df.head()

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


## 1. Select and combine useful content features

In [3]:
features = ["Series_Title", "Genre", "Director", "Star1", "Star2", "Star3"]
df = df[features].copy()

for col in ["Genre", "Director", "Star1", "Star2", "Star3"]:
    df[col] = df[col].fillna("")

df["Combined_Features"] = (
    df["Genre"].str.replace(",", " ", regex=False) + " " +
    df["Director"].str.replace(" ", "_", regex=False) + " " +
    df["Star1"].str.replace(" ", "_", regex=False) + " " +
    df["Star2"].str.replace(" ", "_", regex=False) + " " +
    df["Star3"].str.replace(" ", "_", regex=False)
)

df[["Series_Title", "Combined_Features"]].head()

,Series_Title,Combined_Features
0,The Shawshank Redemption,Drama Frank_Darabont Tim_Robbins Morgan_Freema...
1,The Godfather,Crime Drama Francis_Ford_Coppola Marlon_Brand...
2,The Dark Knight,Action Crime Drama Christopher_Nolan Christi...
3,The Godfather: Part II,Crime Drama Francis_Ford_Coppola Al_Pacino Ro...
4,12 Angry Men,Crime Drama Sidney_Lumet Henry_Fonda Lee_J._C...


## 2. Convert text features to vectors

In [4]:
vectorizer = CountVectorizer()
feature_matrix = vectorizer.fit_transform(df["Combined_Features"])
feature_matrix.shape

(1000, 2606)

## 3. Calculate cosine similarity

In [5]:
similarity_matrix = cosine_similarity(feature_matrix)
similarity_matrix.shape

(1000, 1000)

## 4. Recommendation function

In [6]:
def recommend_movies(movie_title, n_recommendations=6):
    matches = df.index[df["Series_Title"] == movie_title].tolist()
    if not matches:
        raise ValueError(f"Movie not found: {movie_title}")

    movie_index = matches[0]
    scores = list(enumerate(similarity_matrix[movie_index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommendations = scores[1:n_recommendations + 1]
    indices = [idx for idx, _ in recommendations]

    result = df.iloc[indices][["Series_Title", "Genre", "Director"]].copy()
    result["Similarity"] = [score for _, score in recommendations]
    return result.reset_index(drop=True)

In [7]:
recommend_movies("The Godfather", 6)

,Series_Title,Genre,Director,Similarity
0,The Godfather: Part II,"Crime, Drama",Francis Ford Coppola,0.666667
1,The Godfather: Part III,"Crime, Drama",Francis Ford Coppola,0.666667
2,Scarface,"Crime, Drama",Brian De Palma,0.500000
3,Apocalypse Now,"Drama, Mystery, War",Francis Ford Coppola,0.462910
4,Heat,"Crime, Drama, Thriller",Michael Mann,0.462910
5,Dog Day Afternoon,"Biography, Crime, Drama",Sidney Lumet,0.462910
